# CLOY Processing

This notebook is for you to work with the CLOY data folder.

#### reading the jsonl

In [28]:
import json
loadeddata = []
with open("data/RedditExplorer/cloy_chronological.jsonl", "r", encoding="utf-8") as file:
    for lines in file:
        if lines.strip():
            loadeddata.append(json.loads(lines))
        

##### getting the text i want from each post via the title, body/selftext, and the comments

In [29]:
l = []
for line in loadeddata:
    text =line["title"]+ " " + line["selftext"]
    for comment in line["comments"]:
        text =text + " " + comment["comment_body"]
    l.append(text.lower())

### creating list of vocab(unique words)

In [39]:
vocabulary = set()
for doc in l:
    for word in doc.split():
        vocabulary.add(word)

Vocabularylist = sorted(list(vocabulary))

### term freq as seen in slide 24

In [40]:
vectors = []
for new in l:
    words = new.split()
    #to store frequency for later
    counts = {}
    #got help from google for the .get idea below bc my computer was crashing and i needed to find a wind to process the data
    for w in words:
        counts[w] = counts.get(w, 0) + 1
    vector=[]
    for term in Vocabularylist:
        vector.append(counts.get(term, 0))
    vectors.append(vector)

In [41]:
import numpy as np
from numpy.linalg import norm
import pandas as pd

In [42]:
def cosineSimilarity(v1, v2, norm1=None, norm2=None):
    if norm1 is None:
        norm1 = norm(v1)
    if norm2 is None:
        norm2 = norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
        #avoiding dividing by zero
    return np.dot(v1, v2) /(norm1 * norm2)

In [43]:
#after much trial and error, I went to google for help: why couldn't I run thismore simply and it be fine?
#the amount of info in the file is so large that it crashes and takes forever to compare a post to every other post at a time
#so, google gave me the idea to convert all the vectors into one big array and use numpyro to calculate all the sim scores all at once
def cosineSimilarity(v1, allvectors):
    dot_products = np.dot(allvectors, v1)
    magnitudes = norms * norm(v1)
    magnitudes[magnitudes == 0] = 1.0
    return dot_products / magnitudes

V = np.array(vectors, dtype=np.float32)
lengths = norm(V, axis=1)
lengths[lengths == 0] = 1.0 #to prevent errors with division

simscores = []
for i in range(len(V)):
    dots = np.dot(V, V[i]) # computing w all other rows
    row = dots / (lengths * lengths[i])
    simscores.append(row.tolist())


df = pd.DataFrame(simscores)
df.to_csv("cloycosinesimilarities.csv", index=False)

## Q3

In [44]:
ids = [thread.get("post_id", "") for thread in loadeddata]
titles = [thread.get("title", "") for thread in loadeddata]
#"" suggested by friend to prevent crashing if a post doesnt have a little or postid

In [45]:
def top5(id):
    if id not in ids:
        print("thread id not found")
    indxofid = ids.index(id)
    scores = df.values[indxofid]
    ranked = []
    for i in range(len(scores)):
        score = scores[i]
        ranked.append((score, i))
    ranked.sort(reverse=True)
    for i in range(1, 6):
        item = ranked[i]
        score = item[0]
        idx =item[1]
        print("Thread ID:", ids[idx])
        print("Title:", titles[idx])
        print("Cos similarity score:", round(score, 4))
        
    

# Q4

In [46]:
top5(ids[1])

Thread ID: i42ez5
Title: How come you started watching it?
Cos similarity score: 0.9323
Thread ID: qg3ups
Title: KDrama Virgin hit by CLOY tsunami
Cos similarity score: 0.9156
Thread ID: 1ci4cmh
Title: I have just finished binge-watching Crash Landing On You. I am overwhelmed.
Cos similarity score: 0.9155
Thread ID: 17tpx8l
Title: I just finished “Crash Landing On You” and I will never be the same.
Cos similarity score: 0.9102
Thread ID: gypswi
Title: How has watching CLOY made an impact in your life?
Cos similarity score: 0.9096


#### I went to the subreddit explorer and entered this titles to cross-check the similarity and can confirm they all were about the same topic: an all-consuming love for the show and almost like a new love for life (people were very enthusiastic).

#### Q5:
#### I think the filler words are impacting the results. Especially with them being as common as they are, I understand why it would be good practice to remove them from these calculations. If you get rid of them, it will get more precise answers that are more on topic because its searching for overlap in substance not the fluff, I think that's a good way to put it.